# TFG StatsBomb: impacto, perfiles, similitud y encaje

Notebook actualizado para reproducir el pipeline que consume la app Streamlit.

Este notebook no contiene credenciales. Si necesitas descargar datos de StatsBomb, define antes estas variables de entorno:

```powershell
$env:SB_USERNAME="tu_usuario"
$env:SB_PASSWORD="tu_password"
```

Flujo actual:

1. Cargar o construir el dataset maestro de LaLiga 2024/2025.
2. Calcular subimpactos e impactos agregados por rol.
3. Hacer PCA por rol con las variables vigentes.
4. Aplicar K-Means con los valores de `k` elegidos manualmente tras la comparativa provisional.
5. Asignar etiquetas futbolisticas finales a los clusters.
6. Construir similitud tactica entre equipos mediante coseno.
7. Generar rankings y ejemplos de encaje jugador-club.

In [ ]:
from pathlib import Path
import importlib
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

import tfg_pipeline
importlib.reload(tfg_pipeline)
from tfg_pipeline import *

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)

## 1. Configuracion vigente del modelo

Estas tablas son la fuente de verdad del trabajo: subimpactos, variables de estilo para similitud coseno, `k` por rol y etiquetas finales de perfiles.

In [ ]:
subimpact_rows = []
for subgroup, metrics in METRIC_SUBGROUPS.items():
    subimpact_rows.append({
        "subimpacto": subgroup,
        "n_variables": len(metrics),
        "variables": ", ".join(metrics),
    })
display(pd.DataFrame(subimpact_rows))

cluster_rows = []
for role, k in ROLE_CLUSTER_K.items():
    cluster_rows.append({
        "role": role,
        "k_final": k,
        "etiquetas_finales": ", ".join(CLUSTER_LABEL_RULES.get(role, {}).keys()),
        "subimpactos_perfilado": ", ".join(ROLE_GROUPS.get(role, [])),
    })
display(pd.DataFrame(cluster_rows).sort_values("role"))

team_style_df = pd.DataFrame({"variable_similitud_coseno": TEAM_DIRECT_STYLE_FEATURES})
display(team_style_df)

## 2. Carga de datos

Primero intenta leer `data/processed/master_laliga_players.parquet`. Si no existe y hay credenciales de StatsBomb en variables de entorno, descarga y construye el dataset maestro.

In [ ]:
master_path = PROCESSED_DIR / "master_laliga_players.parquet"

if master_path.exists():
    master_df = pd.read_parquet(master_path)
    master_df = ensure_model_role(master_df)
else:
    if not os.getenv("SB_USERNAME") or not os.getenv("SB_PASSWORD"):
        raise RuntimeError("No existe master_laliga_players.parquet y faltan SB_USERNAME/SB_PASSWORD en variables de entorno.")
    player_stats, team_stats, competition_info = load_statsbomb_laliga(force_download=False)
    master_df = build_master_dataset(player_stats, team_stats)
    master_df = ensure_model_role(master_df)
    master_df.to_parquet(master_path, index=False)

pd.Series(resumen_dataset(master_df), name="resumen")

## 3. Diagnostico exploratorio basico

Revision rapida de muestra, minutos y disponibilidad de variables antes de modelar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
order = master_df["role"].value_counts().index
sns.countplot(data=master_df, x="role", order=order, color="#2f6f73", edgecolor="black", ax=axes[0])
axes[0].set_title("Jugadores por rol modelado")
axes[0].set_xlabel("Rol")
axes[0].set_ylabel("Jugadores")

sns.histplot(master_df["Minutes"].dropna(), bins=30, color="#d71920", ax=axes[1])
axes[1].set_title("Distribucion de minutos")
axes[1].set_xlabel("Minutos")
plt.tight_layout()
plt.show()

all_model_groups = sorted({group for groups in ROLE_GROUPS.values() for group in groups})
feature_cols = get_model_feature_columns(master_df, groups=all_model_groups, exclude_team_context=True)
print(f"Variables disponibles para modelado: {len(feature_cols)}")
display(detectar_correlaciones_fuertes(master_df, feature_cols).head(25))

## 4. PCA por rol

El PCA se ajusta por rol para comparar jugadores dentro de familias funcionales comparables.

In [ ]:
pca_results = fit_all_role_pcas(master_df)

pca_summary = []
for role, result in pca_results.items():
    pca_summary.append({
        "role": role,
        "n_jugadores": len(result.scores),
        "n_features": len(result.features),
        "n_componentes": len(result.explained_variance),
        "varianza_acumulada": round(float(result.explained_variance.sum()), 3),
        "features": ", ".join(result.features),
    })
display(pd.DataFrame(pca_summary).sort_values("role"))

## 5. Clustering oficial por rol

Los valores de `k` ya no se eligen automaticamente en produccion. Se fijan tras la comparativa realizada en la app:

- Centrales: `k=3`.
- Delanteros, extremos, laterales, mediocentros y porteros: `k=2`.

La tabla de silhouette se conserva solo como diagnostico.

In [ ]:
players_modeled, cluster_results = fit_all_clusters(master_df, pca_results)
players_scored = calculate_impact_scores(players_modeled)
players_scored.to_parquet(PROCESSED_DIR / "players_scored.parquet", index=False)

cluster_summary = []
for role, result in cluster_results.items():
    labels = players_scored.loc[players_scored["role"].eq(role), "cluster_label"].value_counts().to_dict()
    cluster_summary.append({
        "role": role,
        "k_aplicado": result.k,
        "k_configurado": ROLE_CLUSTER_K.get(role),
        "jugadores": len(result.labels),
        "perfiles": labels,
    })
display(pd.DataFrame(cluster_summary).sort_values("role"))

for role, result in cluster_results.items():
    print(f"\nSilhouette diagnostico {role}")
    display(result.silhouette_table)

## 6. Interpretacion de clusters

Importancia Random Forest y perfiles medios estandarizados. Esta parte sirve para justificar que las etiquetas futbolisticas son coherentes.

In [ ]:
role_to_review = "DEF"
if role_to_review in cluster_results:
    result = cluster_results[role_to_review]
    display(result.feature_importance.head(20).rename("importancia_RF").to_frame())
    display(result.cluster_profiles_z.T.head(30))

profile_counts = (
    players_scored.groupby(["role", "profile_cluster", "cluster_label"], as_index=False)
    .agg(jugadores=("Name", "count"), minutos_medios=("Minutes", "mean"), impacto_medio=("impacto_global", "mean"))
)
profile_counts["minutos_medios"] = profile_counts["minutos_medios"].round(1)
profile_counts["impacto_medio"] = profile_counts["impacto_medio"].round(2)
display(profile_counts.sort_values(["role", "cluster_label"]))

## 7. Impactos y rankings

Los impactos son percentiles agregados por rol. El impacto global usa pesos por rol y, cuando hay perfil especifico, pesos por perfil.

In [ ]:
ranking_cols = existing_columns(
    players_scored,
    ["Name", "Team", "role", "cluster_label", "Minutes", "impacto_global", "impacto_ofensivo", "impacto_asociativo", "impacto_defensivo", "impacto_porteria"],
)
display(players_scored[ranking_cols].sort_values("impacto_global", ascending=False).head(25))

role = "MED"
role_ranking = top_players_by_role(players_scored, role, n=20)
display(role_ranking)

plt.figure(figsize=(10, 6))
sns.barplot(data=role_ranking.sort_values("impacto_global"), x="impacto_global", y="Name", hue="Team", dodge=False, edgecolor="black")
plt.title(f"Top impacto global | {role}")
plt.xlabel("Impacto global")
plt.ylabel("")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. Similitud tactica entre equipos

La similitud coseno usa `TEAM_DIRECT_STYLE_FEATURES`: variables colectivas directas de estilo de equipo, no metricas puras de rendimiento ni puntos.

In [ ]:
team_similarity = calculate_team_similarity(players_scored)

display(team_similarity.team_profiles.head())
display(team_similarity.cosine_matrix.round(3))

if "Osasuna" in team_similarity.cosine_matrix.index:
    display(similar_teams(team_similarity, "Osasuna", n=8).rename("similitud_coseno").to_frame())

data = team_similarity.pca_map.reset_index(names="Team")
plt.figure(figsize=(10, 7))
sns.scatterplot(data=data, x="Team_PC1", y="Team_PC2", hue="cluster", s=160, edgecolor="black", palette="Set2")
for _, row in data.iterrows():
    plt.text(row["Team_PC1"] + 0.03, row["Team_PC2"] + 0.03, row["Team"], fontsize=9)
plt.title("Mapa PCA de estilos de equipo")
plt.tight_layout()
plt.show()

## 9. Encaje jugador-club

El encaje combina impacto individual, similitud de contexto, necesidad del equipo por subimpactos del rol, ajuste al rol del equipo destino, edad y viabilidad economica cuando hay datos Transfermarkt.

In [ ]:
players_with_market = add_economic_data(players_scored)

target_team = "Osasuna"
selected_profile = "Central dominador de area"
fit = player_team_fit(
    players_with_market,
    target_team=target_team,
    cluster_label=selected_profile,
    top_n=20,
    include_economic="market_value_eur" in players_with_market.columns,
)
fit_cols = existing_columns(
    fit,
    ["Name", "Team", "role", "cluster_label", "Minutes", "impacto_global", "fit_score", "context_fit", "team_need_fit", "role_fit", "economic_fit", "age_fit", "market_value_million_eur"],
)
display(fit[fit_cols])

## 10. Guardado final para la app

La app lee principalmente `data/processed/players_scored.parquet`. Si cambias el pipeline, vuelve a ejecutar hasta esta celda.

In [ ]:
players_scored.to_parquet(PROCESSED_DIR / "players_scored.parquet", index=False)
master_df.to_parquet(PROCESSED_DIR / "master_laliga_players.parquet", index=False)
print("Guardado:")
print(PROCESSED_DIR / "players_scored.parquet")
print(PROCESSED_DIR / "master_laliga_players.parquet")

## Notas metodologicas actualizadas

- Los clusters describen perfiles, no calidad absoluta.
- El `k` final es una decision metodologica fijada tras comparar `k=2` y `k=3` por rol.
- Los impactos se mantienen en escala percentil 0-100 sin normalizar el maximo a 100 para evitar exagerar diferencias reales.
- La similitud de equipos usa variables colectivas de estilo y se calcula con coseno tras imputacion y estandarizacion.
- Las recomendaciones de encaje son una ayuda de scouting, no una prediccion causal de rendimiento futuro.